# Assignment: Capacity under Demand Uncertainty

**Course:** Optimization of Energy Systems (15s09)  
**Authors:** Max Glanville & Andres Garcia Angulo  
**Stack:** Julia · JuMP · HiGHS (Gurobi optional later)

A hyperscaler must choose a grid connection offer and optional onsite gas capacity **before** knowing AI compute demand and revenue. This notebook builds that model in layers. **Layer 1** (runnable below) is a plain LP: one year / one contract / known demand — *given a grid connection with some curtailed hours, how much gas should I build?*

## Learning goals

By the end of this assignment you should be able to:

1. Solve **Layer 1**: a deterministic LP for gas capacity under known demand and curtailable grid imports.
2. Frame the full problem as a **two-stage stochastic MILP** (here-and-now vs wait-and-see).
3. Encode **firm vs flexible** grid offers and optional **onsite CCGT** capacity in JuMP.
4. Use **mock campus / workload / plant data**, then replace it with real sources (PJM, Nvidia, EIA, etc.).
5. Compare contracts on **expected NPV** and probe thresholds (curtailment severity, revenue per MW, workload mix).
6. State clearly what the model leaves out (batteries, endogenous prices, GPU depreciation, …).

## Problem statement

A hyperscaler wants its campus earning revenue as soon as possible, but the grid cannot supply firm power quickly. It can:

- connect sooner if it agrees to **cut consumption in peak hours** (flexible / curtailable), or
- wait for **firm** grid service, and/or
- **build onsite generation** (expensive; may sit idle much of the year).

Both grid and generation commitments are made years ahead and are hard to reverse, while the value of compute is uncertain (workload mix, utilisation, chip efficiency).

**Research question:** How should a profit-maximising hyperscaler secure power for a new AI data-centre campus (build onsite generation, accept a curtailable grid connection, wait for firm grid service, or a mix) when it must commit before it knows how much computing work will arrive, what kind, and what it will earn?

## Environment & solvers

Activate this folder’s project, then load JuMP. **Layer 1 uses HiGHS** (open-source LP/MIP) so the notebook solves without a commercial license. **Gurobi** remains optional for later layers once licensed (`gurobi.lic` / academic license).

In [ ]:
using Pkg
Pkg.activate(".")
Pkg.instantiate()

using JuMP
using HiGHS
using DataFrames
# using Gurobi   # optional later: uncomment when licensed Gurobi is installed

println("Julia ", VERSION)
println("JuMP + HiGHS ready (Gurobi optional for later layers).")

---

## Layer 1 — One year, one contract, known demand (plain LP)

**Question:** given a grid connection with some curtailed hours, how much gas should I build?

### Decision variables
| Variable | Meaning |
|----------|--------|
| $G \ge 0$ | Gas capacity built (MW), first-stage |
| $x_t$ | GPUs busy in hour $t$ |
| $g_t$ | Gas output in hour $t$ (MW) |
| $q_t$ | Grid imports in hour $t$ (MW) |

### Objective
$$
\max \sum_t \bigl[ r \cdot x_t - \lambda_t \cdot q_t - c_G \cdot g_t \bigr] - I_G \cdot G
$$
Revenue from compute, minus grid energy, gas fuel, and annualised (here: horizon-scaled) gas investment.

### Constraints
| # | Constraint | Meaning |
|---|------------|--------|
| 1 | $x_t \le A_t$ | Can't serve more work than arrives |
| 2 | $x_t \le C$ | Can't use more GPUs than you own |
| 3 | $\mathrm{PUE}\cdot(e\,x_t + e_0\,(C-x_t)) \le g_t + q_t$ | Power drawn ≤ power supplied |
| 4 | $g_t \le \alpha_t\, G$ | Gas output ≤ built capacity × availability |
| 5 | $q_t \le P\,(1-E_t)$ | Grid imports capped; zero when $E_t=1$ |

### Layer 1 mock data

Small horizon (48 h = 2 representative days) with **peak curtailment** (`E_t = 1`) on a few evening hours so the trade-off between grid and gas is visible. Parameters are toy-scaled for the short horizon; replace with real campus / tariff / plant data later.

In [ ]:
# --- Layer 1 horizon & campus ---
T = 1:48                      # two representative days (hours)
C = 1_000.0                   # GPUs owned
PUE = 1.25
e = 0.0017                    # MW per busy GPU
e0 = 0.0003                   # MW per idle GPU
P = 1.5                       # MW grid import cap when not curtailed

# Economics (toy units consistent over this 48 h horizon)
r = 0.12                      # $/GPU-hour revenue
c_G = 25.0                    # $/MWh gas fuel (≈ heat rate × fuel price)
# Annualised capex ~ $120k/MW-year → scale to 48/8760 of a year
I_G = 120_000.0 * (length(T) / 8_760.0)   # $ per MW for this horizon

# Time series
A = Dict{Int,Float64}()       # arriving work (GPU-hours demand)
λ = Dict{Int,Float64}()       # grid price $/MWh
α = Dict{Int,Float64}()       # gas availability
E = Dict{Int,Float64}()       # curtailment indicator (1 = curtailed)

for t in T
    hour_of_day = mod(t - 1, 24)  # 0..23
    # Demand: higher daytime, lower night
    A[t] = hour_of_day in 8:21 ? 950.0 : 400.0
    # Grid price: mild peak in late afternoon
    λ[t] = hour_of_day in 16:20 ? 80.0 : 40.0
    α[t] = 0.95
    # Curtail evenings on both days (and one midday spike on day 2)
    E[t] = (hour_of_day in 17:20) || (t == 36) ? 1.0 : 0.0
end

n_curtailed = count(t -> E[t] == 1.0, T)
println("Hours: ", length(T), " | Curtailed (E_t=1): ", n_curtailed)
println("I_G (horizon-scaled) = ", round(I_G; digits=2), " $/MW")

DataFrame(
    t = collect(T),
    A_t = [A[t] for t in T],
    λ_t = [λ[t] for t in T],
    E_t = [Int(E[t]) for t in T],
    α_t = [α[t] for t in T],
)

### Build & solve Layer 1 (JuMP + HiGHS)

In [ ]:
"""
build_layer1_model(T, A, λ, α, E; C, PUE, e, e0, P, r, c_G, I_G, optimizer)

Plain LP: choose gas capacity G and hourly dispatch (x, g, q) under known demand
and a curtailable grid connection.
"""
function build_layer1_model(
    T, A, λ, α, E;
    C, PUE, e, e0, P, r, c_G, I_G,
    optimizer = HiGHS.Optimizer,
)
    m = Model(optimizer)
    set_silent(m)

    @variable(m, G >= 0)                 # MW gas capacity
    @variable(m, x[T] >= 0)              # GPUs busy
    @variable(m, g[T] >= 0)              # gas MW
    @variable(m, q[T] >= 0)              # grid MW

    for t in T
        @constraint(m, x[t] <= A[t])                                    # (1)
        @constraint(m, x[t] <= C)                                       # (2)
        @constraint(m, PUE * (e * x[t] + e0 * (C - x[t])) <= g[t] + q[t])  # (3)
        @constraint(m, g[t] <= α[t] * G)                                # (4)
        @constraint(m, q[t] <= P * (1 - E[t]))                          # (5)
    end

    @objective(m, Max,
        sum(r * x[t] - λ[t] * q[t] - c_G * g[t] for t in T) - I_G * G
    )
    return m
end

layer1 = build_layer1_model(
    T, A, λ, α, E;
    C=C, PUE=PUE, e=e, e0=e0, P=P, r=r, c_G=c_G, I_G=I_G,
)
optimize!(layer1)

@assert termination_status(layer1) == MOI.OPTIMAL "Layer 1 did not solve to optimality"

G_star = value(layer1[:G])
obj_star = objective_value(layer1)
x_star = value.(layer1[:x])
g_star = value.(layer1[:g])
q_star = value.(layer1[:q])

println("=== Layer 1 results ===")
println("Status:     ", termination_status(layer1))
println("G* (MW):    ", round(G_star; digits=4))
println("Objective:  ", round(obj_star; digits=2))
println("Mean x_t:   ", round(sum(x_star[t] for t in T) / length(T); digits=1), " GPUs")
println("Mean g_t:   ", round(sum(g_star[t] for t in T) / length(T); digits=4), " MW")
println("Mean q_t:   ", round(sum(q_star[t] for t in T) / length(T); digits=4), " MW")

In [ ]:
# Summary table (first day + curtailment flag)
summary = DataFrame(
    t = collect(T),
    E_t = [Int(E[t]) for t in T],
    A_t = [A[t] for t in T],
    x_t = [round(x_star[t]; digits=1) for t in T],
    g_t = [round(g_star[t]; digits=4) for t in T],
    q_t = [round(q_star[t]; digits=4) for t in T],
    λ_t = [λ[t] for t in T],
)

println("\nHourly dispatch (curtailed hours should show q_t ≈ 0 and g_t covering load):")
display(summary)

# Tiny text plot: gas vs grid over the horizon
println("\nGas g_t (█) vs grid q_t (░) — scaled bars")
scale = 20.0  # MW → bar chars
for t in T
    ng = max(0, round(Int, g_star[t] * scale))
    nq = max(0, round(Int, q_star[t] * scale))
    flag = E[t] == 1.0 ? " E" : "  "
    println(lpad(string(t), 3), flag, " |", "█"^ng, "░"^nq)
end

println("\nG* = ", round(G_star; digits=4), " MW  |  objective = ", round(obj_star; digits=2))

---

## Later layers — full two-stage stochastic MILP (assignment Tasks A–E)

Layer 1 fixes demand and a single curtailable contract. The full research model adds discrete **firm vs flexible** offers, demand scenarios $\omega$, and stage-1 choices before $\omega$ is revealed. Tasks below keep that framing; Layer 1 above is the runnable core you can extend.

## Mock data for later layers (stand-in for appendix tables)

Standard campus: **100 MW IT load**, ~current-generation Nvidia Blackwell-class racks. Two grid offers: **K1 Firm** (later connection, no curtailment) and **K2 Flexible** (earlier connection, peak curtailment). Demand scenarios ω differ by volume / mix multipliers θ_ω with probabilities p_ω.

> Replace these placeholders with real offers and parameters when you complete Task A.

In [ ]:
# --- Sets (toy) ---
K = [:firm, :flexible]          # grid offers
J = [:inference, :agentic, :training]  # workload classes
Ω = [:low, :base, :high]        # demand scenarios
T_full = 1:24                   # representative day for later layers

# --- Grid offers (mock; inspired by proposal Table 2) ---
grid = Dict(
    :firm => (
        lead_years = 4.0,
        curtail_frac = 0.0,
        capacity_mw = 100.0,
        fixed_charge_m_per_year = 12.0,
        energy_price = 45.0,
    ),
    :flexible => (
        lead_years = 1.5,
        curtail_frac = 0.15,
        capacity_mw = 100.0,
        fixed_charge_m_per_year = 6.0,
        energy_price = 40.0,
    ),
)

# --- Campus (mock Table 3) ---
campus = (
    it_load_mw = 100.0,
    n_gpus = 58_800,
    pue = 1.25,
)

# --- Workload / demand uncertainty (mock Table 4) ---
p_ω = Dict(:low => 0.25, :base => 0.50, :high => 0.25)
θ_ω = Dict(:low => 0.70, :base => 1.00, :high => 1.30)
rev_per_mw_year = Dict(
    :inference => 11.0,
    :agentic => 10.0,
    :training => 9.0,
)
flex_share = Dict(
    :inference => 0.20,
    :agentic => 0.40,
    :training => 0.80,
)

# --- Onsite CCGT (mock Table 5) ---
ccgt = (
    capex_per_mw = 1.2e6,
    om_fixed_per_mw_year = 2.0e4,
    heat_rate = 7.0,
    fuel_price = 3.5,
    avail = 0.92,
    life_years = 20,
)

discount_rate = 0.08
horizon_years = 10

DataFrame(
    offer = collect(keys(grid)),
    lead_years = [grid[k].lead_years for k in keys(grid)],
    curtail_frac = [grid[k].curtail_frac for k in keys(grid)],
)

## Model sketch (two-stage stochastic MILP)

### Stage 1 (here-and-now) — before ω is revealed
- Binary (or SOS1) choice of grid offer: $z_k$, $\sum_k z_k = 1$.
- Continuous onsite capacity: $G \ge 0$ (MW of CCGT) — same role as Layer 1’s $G$.

### Stage 2 (wait-and-see) — for each scenario ω, hour t, workload j
- Work processed, grid import, turbine dispatch, unmet / deferred demand, …

### Objective
Maximise **expected discounted profit** (compute revenue − energy − fuel − O&M − annualised capex − grid fixed charges), accounting for lead time before the campus is energised.

### Constraint families (proposal C1–C10)
- Work accounting & GPU capacity caps  
- Power balance (IT × PUE)  
- Turbine limits ≤ built capacity × availability  
- Grid imports only after connection and within offer rules (curtailment windows for flexible)  
- Logic linking $z_k$ to which offer parameters apply  

**Problem class:** two-stage stochastic **MILP** (integer only because of discrete contract choice $z$). Solver: **HiGHS** for development; **Gurobi via JuMP** optional when licensed.

## Task A — Data TODOs

Fill or replace the mock tables. For each item: *what you need*, *why*, *source you will use*.

| ID | TODO | Status |
|----|------|--------|
| A1 | Real grid offers (TX / VA / OR or PJM-style): lead time, curtailment rules, capacity, charges | `[ ]` |
| A2 | Campus IT load, GPU count, PUE, revenue per MW by workload | `[ ]` |
| A3 | Demand scenarios θ_ω, p_ω, and workload mix by ω | `[ ]` |
| A4 | CCGT capex, heat rate, fuel, O&M, lifetime (EIA / OEM) | `[ ]` |
| A5 | Peak-hour mask for flexible offer (which hours curtailed) | `[ ]` |
| A6 | Document units and USD year for every money parameter | `[ ]` |

### Deliverable
Update the dictionaries above (or load CSVs) and briefly cite sources in a markdown cell under this section.

In [ ]:
# Task A helpers — load CSVs once you create them, e.g.:
# using CSV
# offers = CSV.read("data/grid_offers.csv", DataFrame)

function assert_probabilities(p::Dict)
    s = sum(values(p))
    @assert abs(s - 1.0) < 1e-8 "Scenario probabilities must sum to 1 (got $s)"
end

assert_probabilities(p_ω)
println("Mock probabilities OK. Replace with real data for Task A.")

## Task B — JuMP stubs (extend Layer 1 → two-stage MILP)

Build on Layer 1: stage-1 decisions shared across scenarios; stage-2 indexed by ω (and t, j as needed).

> Stub for later layers. Layer 1 above is already solvable.

In [ ]:
"""
build_hyperscaler_model(; optimizer)

Returns a JuMP model with:
  Stage 1: z[k] binary grid-offer choice, G >= 0 onsite MW
  Stage 2: placeholders for dispatch / work variables by ω
  Objective: expected discounted profit (TODO)
  Constraints: stubs for C1–C10 (TODO)

Default optimizer is HiGHS; pass Gurobi.Optimizer when licensed.
"""
function build_hyperscaler_model(; optimizer = HiGHS.Optimizer)
    m = Model(optimizer)
    set_silent(m)

    # --- Stage 1 ---
    @variable(m, z[K], Bin)
    @variable(m, G >= 0)
    @constraint(m, choose_one_offer, sum(z[k] for k in K) == 1)

    # --- Stage 2 placeholders ---
    @variable(m, grid_mw[Ω, T_full] >= 0)
    @variable(m, turb_mw[Ω, T_full] >= 0)
    @variable(m, work[J, Ω, T_full] >= 0)

    for ω in Ω, t in T_full
        @constraint(m, turb_mw[ω, t] <= G * ccgt.avail)
        @constraint(m, grid_mw[ω, t] <= sum(z[k] * grid[k].capacity_mw for k in K))
    end

    @objective(m, Max, 0.0)  # TODO: E_ω[NPV]
    return m
end

model = build_hyperscaler_model()
println(model)
println("Task B: flesh out objective + C1–C10; Layer 1 already demonstrates the gas/grid LP core.")

## Task C — Solve (later layers)

Layer 1 is already solved above with HiGHS. For the full MILP:

In [ ]:
# Task C — full model solve once Task B objective/constraints are complete
# model = build_hyperscaler_model(optimizer = HiGHS.Optimizer)
# # or: using Gurobi; model = build_hyperscaler_model(optimizer = Gurobi.Optimizer)
# optimize!(model)
# println(termination_status(model))
# println("z = ", value.(model[:z]))
# println("G = ", value(model[:G]))

println("Task C: Layer 1 done; uncomment above after Task B is complete.")

## Task D — NPV experiments & write-up

Experimental design from the proposal: compare **firm vs flexible** (and mixes with onsite gas) on expected NPV. Then stress key levers.

Suggested sweeps (start with one variable at a time):

1. Curtailment fraction / peak-hour severity on the flexible offer  
2. Revenue per MW-year  
3. Workload mix (more training vs inference)  
4. Fuel price or CCGT capex  

**Null result to watch for:** firm always beats flexible in every scenario (or the reverse).

### Write-up (markdown below or separate `writeup.md`)
- Optimal stage-1 plan and expected NPV  
- NPV by scenario ω  
- Thresholds where the preferred contract flips  
- ½–1 page interpretation tied to the research question

In [ ]:
# Task D — experiment harness (fill in once full model solves)
function expected_npv_stub(z_choice::Symbol, g_mw::Float64)
    return NaN
end

results = DataFrame(
    offer = Symbol[],
    g_mw = Float64[],
    expected_npv = Float64[],
)
for k in K, g in (0.0, 50.0, 100.0)
    push!(results, (k, g, expected_npv_stub(k, g)))
end
results

### Task D write-up space

*Replace this cell with your findings.*

## Task E — Stretch

Pick **one** (or propose your own) and sketch how you would extend the JuMP model:

- Endogenous **electricity prices** if many campuses connect (not treated as fixed inputs)  
- **Batteries / solar** or multi-site workload shifting  
- **GPU choice** / efficiency as a stage-1 decision  
- Risk measure beyond expected profit (CVaR, worst-case ω)  
- Replace mock offers with a **panel of real US interconnection options**

In [ ]:
stretch_ideas = [
    "CVaR on NPV across Ω",
    "Battery SoC balance + charge/discharge vars",
    "Binary GPU portfolio with different J efficiencies",
]
stretch_ideas

## Out of scope (from the proposal)

Be explicit in your write-up that the baseline model **does not** include:

- Demand-driven transmission upgrades or equilibrium prices  
- Uncertain / stochastic curtailment timing (we assume peak windows known ex ante)  
- Batteries, solar, or spatial flexibility across sites  
- GPU depreciation or efficiency drift over the horizon  
- Over-weighting of extreme scenarios for commercial risk appetite

## Checklist

- [x] **Layer 1** LP runnable with HiGHS (G*, objective, hourly summary)  
- [ ] Learning goals read; research question stated in own words  
- [ ] Env activates; JuMP + HiGHS imports (Gurobi noted if using later)  
- [ ] Task A data TODOs addressed or explicitly deferred with sources  
- [ ] Task B JuMP two-stage stubs filled toward a solvable MILP  
- [ ] Task C full-model solve (or documented license blocker)  
- [ ] Task D NPV table + short write-up  
- [ ] Task E stretch idea noted  
- [ ] Out-of-scope limitations listed  
- [ ] `git push` your work to this repo